# The Imagination Gap, the Trust Gap, and the Case for Design-Led Intervention in UK Paludiculture Adoption: Data Synthesis & Full Statistical Verification

Supporting materials for the MASc Capstone thesis *The Imagination Gap, the Trust Gap, and the Case for
Design-Led Intervention in UK Paludiculture Adoption* (London Interdisciplinary
School, 2026).

This notebook is self-contained. The stakeholder survey data (n = 52) and every
image used in the thesis and workshop are embedded directly in the cells below
as base64 data, so it runs end-to-end in Google Colab without needing any file
uploads.


**Contents**
1. Setup and embedded survey data
2. Sample overview
3. The imagination gap / trust gap matrix (Figure 2) — exact thesis coding
4. Clarity and trust distribution, farmers vs. rest (Figure 3)
5. Messenger trust vs. trust in economic viability (Figure 7)
6. Statistical tests — full verification (S4.1, 4.2.1, 4.2.2, 4.2.4, 4.3)
7. Workshop creative outputs (Figures 1, 4, 5, 6, 8)
8. Workshop photo gallery
9. Closing discussion notes
10. Data provenance, item coding, and correction log


> **A note on item selection.** Two of the survey's items look similar but
> measure different things: **Q10** ("how clear is that picture?", a coarser
> 4-point single-select) and **Q12** ("I can picture what a thriving
> paludiculture landscape would look like", a 5-point Likert item). The
> thesis's Methodology (S3.2) specifies **Q12** for the imaginative-clarity
> construct used throughout Chapter 4, so that's the item used everywhere
> below. Every chart in Sections 3–6 is generated directly from the raw
> item-level survey responses embedded in Section 1, using the exact items
> and coding rules set out in the Methodology. The photographic and drawn
> artefacts in Section 8 (Figures 1, 4, 5, 6, 8) are embedded as images
> because they are photographs and drawings, not data, and cannot be
> regenerated any other way.


In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import io, base64
from IPython.display import Image, display, Markdown

pd.set_option('display.max_columns', 100)
plt.rcParams['figure.dpi'] = 110


In [ ]:
# Embedded stakeholder survey data (n = 52), exported from the original .xlsx
survey_csv = """ID,Start time,Completion time,Email,Name,Last modified time,How would you describe your primary role? (Select all that apply),How long have you worked with or managed land? (Select one),How familiar are you with the term ‘paludiculture’ - farming on rewetted peatlands? (Select one),"Have you come across any images, stories, demonstrations, or media that made wet-farming feel desirable or worth pursuing rather than just necessary?","When you try to picture a farm or land holding operating on rewetted peatland, how clear is that picture? (Select one)",How much do you trust that paludiculture could be economically viable on land like the land you work with? (Select one),I can picture what a thriving  paludiculture landscape would look  like.,I believe wet land can be genuinely  productive land.,I think wet-farming could work in  the area I know best.,I feel I have enough information to  form a view on paludiculture.,The evidence for paludiculture feels  relevant to my situation.,"What, if anything, would help you picture paludiculture more clearly or make it feel more real?(For example: seeing a working example, talking to someone doing it, reading about it, financial mode...",rewetting,peatland restoration,paludiculture,"If you were explaining wet-farming to a neighbouring farmer who had never heard of it, what would you say? (Write as you would actually speak; the wording matters here.)",Lack of financial incentives or  viable markets,Policy and regulatory uncertainty,Insufficient technical knowledge or  training,The perception that wet land is unproductive,Not being able to picture what it  would look like in practice,Lack of trust that it has been  proven to work,Cultural unfamiliarity in farming  communities,Tenure and ownership structures,Government or DEFRA guidance,"Conservation NGOs (e.g. RSPB,  Wildlife Trusts)",Farmers or land managers already  doing it,Academic research,Industry bodies or land agencies,"How likely are you to engage more actively with paludiculture or wet-farming in the next five years?
Select one.","Do you feel that the way peatland restoration is currently communicated - thelanguage, images, and stories used - helps or hinders the case for change? Why?This is the most important question in ..."
1,2026-06-09 10:19:30,2026-06-09 10:22:06,anonymous,,,Researcher or academic ;,5–14 years,Very familiar,No,Very unclear,Don't trust it at all,Neither,Neither,Agree,Agree,Agree,,Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),Overly academic (not how I’d naturally talk about it),,Major barrier,Minor barrier,Moderate barrier,Moderate barrier,Minor barrier,Major barrier,Major barrier,Moderate barrier,Moderately,Moderately,Moderately,Trust it fully,Moderately,Somewhat likely,
2,2026-06-09 12:09:22,2026-06-09 12:14:40,anonymous,,,NGO or third sector worker;,5–14 years,Moderately familiar,Yes,Somewhat clear,Moderately,Strongly agree,Strongly agree,Agree,Agree,Agree,,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),Farming on peatlands that are wet (and healthy) and that doesn't damage the peat,Moderate barrier,Moderate barrier,Major barrier,Moderate barrier,Minor barrier,Major barrier,Major barrier,Major barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Very likely,"It's very important to also give geographical context when speaking about Paludiculture. It is very European-focused and should not cause confusion with tropical peatlands. It's important to remember that the main thing peatlands needs is to stop drainage, then restoration, then sustainable management. Paludiculture is not always the best or first solution."
3,2026-06-09 12:36:35,2026-06-09 12:54:01,anonymous,,,Conservation or ecological practitioner ;,Less than 5 years,Moderately familiar,Maybe,Somewhat clear,Slightly,Neither,Agree,Neither,Neither,Agree,seeing a working example,Motivating (they point to something I care about),Motivating (they point to something I care about),Motivating (they point to something I care about),growing wetland species rather than traditional crops that require drier soils,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Somewhat unlikely,"I think it works for people who are interested in conservation and wildlife, however it may not be as effective with farmers, who sometimes see it as a threat to their livelihood."
4,2026-06-09 14:56:21,2026-06-09 15:00:58,anonymous,,,Researcher or academic ;,5–14 years,Very familiar,Yes,Vivid and clear,Moderately,Agree,Agree,Neither,Agree,Agree,Time is needed to see whether paludiculture is going to be a beneficial way to changing farming practices. It's too early in the process at the moment.,Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"By draining organic soils, we risk losing the carbon stored in them. Despite this drainage being needed for traditional agriculture, there is a way in which we could balance keeping soils wet while maintaining their ability to hold onto that carbon. This is called wet-farming (or paludiculture). The practice of keeping soils wet while maintaining a livable crop.",Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Minor barrier,Moderate barrier,Moderately,Moderately,Trust it fully,Trust it fully,Moderately,Somewhat unlikely,Peatland restoration is often framed as the saviour. But it is a long process that requires a lot of time and money. What we should also focus on is AVOIDED CONVERSION of existing peatlands to further disturbances such as agriculture.
5,2026-06-09 16:05:37,2026-06-09 16:19:30,anonymous,,,Researcher or academic ;,Less than 5 years,Slightly familiar,Maybe,Very unclear,Slightly,Neither,Neither,Neither,Disagree,Disagree,Seeing an educational video,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),I don’t know enough about it to be able to describe it confidently,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Minor barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderately,Moderately,Trust it fully,Trust it fully,Moderately,Neither likely nor unlikely,"I don’t engage much with content on peatland restoration as my research focuses more on the transition away from peat for horticulture (i.e., supporting the use of other growing media). From what I’ve come across on peatland restoration (mainly academic literature, or wildlife and conservation charity websites, or the IUCN Peatland Programme website), there is emphasis on carbon storage, habitat protection and flood mitigation. If I’m honest, I haven’t paid too much attention to the language around communicating this, I’m looking more for facts."
6,2026-06-10 07:54:39,2026-06-10 08:14:25,anonymous,,,Farmer or agricultural land manager;,25 years or more,Very familiar,No,Somewhat clear,Don't trust it at all,Strongly disagree,Strongly agree,Neither,Strongly agree,Agree,"I have seen it and been to working demonstrations or establishment. Underwhelmed is an understatement. Bunch of eco mentalist with very little real world experience and being paid to just do it is mental. There is no learning when they have no skin in the game, they do it for the process of actually doing it. A shit show driven by government funding all on the alter of net zero which is fundamentally flawed.",Overly academic (not how I’d naturally talk about it),Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Basically ruining land which could be productive if a new drainage scheme was installed. But that’s not “new” and “on trend”. The name Paludaculture is horrendous also!,Not a barrier,Not a barrier,Not a barrier,Not a barrier,Not a barrier,Not a barrier,,Not a barrier,Don’t trust it at  all,Don’t trust it at  all,Moderately,Don’t trust it at  all,Don’t trust it at  all,Somewhat unlikely,"It should be called Peatland degradation. It’s lazy land agents and environmentalists ruining land for future generations. Who in their right mind would sown their land with rushes, it’s mental! The only people attracted to it are land owners who aren’t farmers and have no idea what to do with their land. Government bodies are not to be trust either. Why would you wet your land further to them have something grow which they then exclaim needs protecting and hey presto you’ve lost control of your land forever. The crazy miss harvesting idea instead of peat. They forget to tell you it needs overhead irrigation, so basically looking for veg land growers. Absolutely bonkers to expect large scale uptake. It’s preposterous. It’s only viable whilst the government is funding it all, so fringe industry is just harvesting tax payers money and looking for patsies to give up land for them to play on. Compete chancers the lot of them, and when the free money runs out they’ll be off looking for more free money and the land owner left with rushes which are a bloody nuisance. I mean who actually gives a shit about they’re puffer jacket filled with rush seeds, other than middle England ladies who brunch!! The world’s in a mess, the country’s lead by donkeys, and these lads are putting rushes in jackets! Good one 🤦‍♂️"
7,2026-06-10 08:46:16,2026-06-10 09:29:18,anonymous,,,Farmer or agricultural land manager;,25 years or more,Slightly familiar,No,Vivid and clear,Don't trust it at all,Agree,Disagree,Disagree,Neither,Disagree,"A working economically viable example. Reliance on a scheme, handout or subsidy does not make it sustainable",Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Letting the moors go to rack and ruin,Major barrier,Major barrier,Moderate barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Don’t trust it at  all,Don’t trust it at  all,Moderately,Slightly,Slightly,Somewhat unlikely,"I fail to understand the goal of all these ideas, and cannot shake the feeling of being a landowning pawn in a never ending game of environmental and political chess. I have fiscal and emotional investment in my farm on the Somerset Levels. The levels were drained hundreds of years ago to create productive farmland. Local towns and villages were all a consequence of that agricultural productivity. Society has moved on, but nonetheless the capital value of that land has been set on the basis of that productivity. ""Restoring"" the land to some figurative time in the past because of pressure from transitory environmental pressure groups or policy seems ill thought out. More pockets of high water table areas, within a complex network of rhynes and watercourses designed to remove water, will cause more conflict between residential and agricultural interests. The drainage in the levels is already a messy conflict between many stakeholders with entirely different agendas: EA, IDB's, Councils, Landowners, Wildlife groups, Highways, Developers etc. I cannot immediately seem an oportunity for income from growing reeds, rushes etc without some form of handout, the longevity of which is traditionally questionable. This is a contrived market place. Adding in conflicting interests of different government bodies, I can only see the landowner losing out. I have expereince of Natural England trying to re-categorise some land after a land management scheme ended,  effectively preventing it being used for intensive agricultre again, even though thay had nothing to do with the original scheme. Unitended consequences and overreach. Good luck, I shall get off my soapbox now!"
8,2026-06-10 09:41:46,2026-06-10 09:47:18,anonymous,,,NGO or third sector worker;,Less than 5 years,Very familiar,Yes,Somewhat clear,Moderately,Agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Robust evidence bringing in all of those examples - and evidence that spans a number of years,Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"It's growing crops that are suitable and profitable for wet, or re-wetted land",Major barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Moderately,Trust it fully,Trust it fully,Moderately,Moderately,Very likely,"I think it could be better - it's needs to be more ground up talking - from the actual farmers who know and care about the land, rather than scientists and policy makers telling people what they should do."
9,2026-06-10 10:04:40,2026-06-10 10:07:23,anonymous,,,Farmer or agricultural land manager;,25 years or more,Slightly familiar,No,Very unclear,Don't trust it at all,Strongly disagree,Strongly disagree,Strongly disagree,Agree,Strongly disagree,,Overly academic (not how I’d naturally talk about it),Overly academic (not how I’d naturally talk about it),Overly academic (not how I’d naturally talk about it),It's a load of bollocks,Major barrier,Major barrier,Minor barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Minor barrier,Don’t trust it at  all,Don’t trust it at  all,Slightly,Don’t trust it at  all,Don’t trust it at  all,Somewhat unlikely,
10,2026-06-10 11:19:09,2026-06-10 11:28:15,anonymous,,,Farmer or agricultural land manager;,25 years or more,Very familiar,Yes,Vivid and clear,Moderately,Strongly agree,Agree,Agree,Agree,Agree,Seeing it in practice backed up by evidence and financial projections,Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),It’s raising the water table to stop the loss of the peat and reduce carbon emissions whilst still growing a crop which is bringing in an income to the farm,Moderate barrier,Moderate barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderately,Moderately,Trust it fully,Moderately,Moderately,Very likely,"There needs to be more farmers taking part and being at the front of media talking and showing what they are doing if we are to encourage other farmers to adopt a different way of farming. Sadly many farmers are resistant to being told how to do things by environmental organisations. They will, however, listen to other farmers. Especially if they see paludiculture as a financially viable prospect within their current farming operations. It’s very important to let farmers know that they are not being told to stop producing food but that paludiculture could be a solution for those blocks of land which are difficult to farm."
11,2026-06-10 22:13:41,2026-06-10 22:18:57,anonymous,,,Conservation or ecological practitioner ; NGO or third sector worker;,15–24 years,Not at all familiar,No,Very unclear,Slightly,Neither,Agree,Agree,Disagree,Agree,All of above,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),No idea,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Don’t trust it at  all,Don’t trust it at  all,Don’t trust it at  all,Don’t trust it at  all,Don’t trust it at  all,Very likely,There is a bigger picture of change resistance
12,2026-06-11 11:40:54,2026-06-11 11:48:07,anonymous,,,Conservation or ecological practitioner ;,25 years or more,Very familiar,Yes,Vivid and clear,Trust it fully,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Working examples are key,Motivating (they point to something I care about),Motivating (they point to something I care about),Motivating (they point to something I care about),How do you see the future of your soils?  Have you heard of farmers who are still growing crops but on wetter land so that they have less drainage costs and a healthier more resilient soil resource.,,,,,,,,,Trust it fully,Moderately,Moderately,Trust it fully,Moderately,Very likely,"much better now e.g. re-peat, sw peat partnership"
13,2026-06-11 11:46:40,2026-06-11 11:56:56,anonymous,,,NGO or third sector worker;,5–14 years,Slightly familiar,No,Very unclear,Slightly,Disagree,Neither,Disagree,Strongly disagree,Neither,Seeing an example of a paludiculture farm to follow,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),Come and see how it works. We can still produce carrots and what not. The soil is just wet and the tractor is not used anymore.,Major barrier,Moderate barrier,Moderate barrier,Minor barrier,Moderate barrier,Major barrier,Major barrier,Minor barrier,Trust it fully,Moderately,Moderately,Trust it fully,Slightly,Neither likely nor unlikely,Peatland communications helps. Especially if it is visual so you help people seeing the future.
14,2026-06-11 11:57:57,2026-06-11 12:01:47,anonymous,,,Researcher or academic ; Other; NGO or third sector worker;,Less than 5 years,Moderately familiar,Yes,Very unclear,Moderately,Disagree,Strongly agree,Agree,Agree,Agree,More real life UK examples,Motivating (they point to something I care about),Technical (useful but not emotionally engaging),Motivating (they point to something I care about),,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderately,Moderately,Trust it fully,Moderately,Slightly,Very likely,Separated from humans. Wet farming can bridge the gap.
15,2026-06-11 11:56:37,2026-06-11 12:03:55,anonymous,,,Conservation or ecological practitioner ;,5–14 years,Slightly familiar,Yes,Very unclear,Don't trust it at all,Neither,Agree,Strongly disagree,Disagree,Neither,"Site visits and open days, it can be quite abstract so seeing working examples would definitely help",Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),"Explaining the rational, why you should raise water tables. Some examples I’d heard of, try to engage them with relevant information. Would use words like emissions reduction, water retention",Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderately,Trust it fully,Trust it fully,Trust it fully,Slightly,Neither likely nor unlikely,When the term rewilding is used incorrectly it confuses people about restoration. The supporting comms need to be more proactive in challenging misinformation.
16,2026-06-11 15:57:52,2026-06-11 16:03:51,anonymous,,,Other;,5–14 years,Very familiar,Yes,Vivid and clear,Moderately,Agree,Agree,Agree,Strongly agree,Agree,Farmers adopting paludiculture and demonstrating the benefits to other farmers,Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),"Working with water water not against it to protect soil and nature whilst providing a stable, productive and profitable crop",Major barrier,Moderate barrier,Major barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderately,Moderately,Trust it fully,Slightly,Moderately,Very likely,If we are trying to get more farmers to grow at higher water tables we need farmers to communicate the benefits. The ghg benefits will not drive adoption as a narrative alone
17,2026-06-11 16:09:51,2026-06-11 16:19:47,anonymous,,,Researcher or academic ;,5–14 years,Very familiar,Maybe,Somewhat clear,Slightly,Neither,Agree,Agree,Agree,Neither,"Working examples - Countryfile has covered some paludiculture, but there needs to be wider media outreach to showcase the practice.",Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),It’s where you can raise the water table on your land to lock in the carbon whilst focussing your framing practice at the very surface for more sustainable outcomes.,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Slightly,Moderately,Trust it fully,Trust it fully,Moderately,Somewhat likely,"I see peatland science as a “toddler”. There’s lots of unanswered questions, and individuals with somewhat “powerful” roles who often see their practice as the most appropriate because they “have done it for 20 years”. We need to become more adaptive, give innovation a real chance, and talk more in “net benefit” from such work. Ideally, this includes all the ecosystem services, not just talking carbon. In terms of language, peatlands have always had it difficult: “i’m bogged down”, “i feel peated out today”, “im just going to the bog (loo)” - Ideally, we need to change langue to accommodate peatlands as a positive thing. Within the community, I’d appreciate seeing more efforts with the youth. Moors for the Futures approach with influencers was actually really unique and impactful. We need ways to communicate the science, incorporating art, heritage, and today’s modern communications."
18,2026-06-11 16:29:17,2026-06-11 16:34:27,anonymous,,,Conservation or ecological practitioner ;,25 years or more,Slightly familiar,Maybe,Very unclear,Slightly,Disagree,Neither,Disagree,Strongly disagree,Disagree,Seeing it on the ground,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),Allowing water to move more naturally across the landscape,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderately,Moderately,Trust it fully,Moderately,Moderately,Neither likely nor unlikely,Alienates current farming practice
19,2026-06-11 17:35:20,2026-06-11 17:41:37,anonymous,,,Researcher or academic ;,5–14 years,Very familiar,No,Somewhat clear,Moderately,Agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,,Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),,Major barrier,Major barrier,Major barrier,Not a barrier,Moderate barrier,Major barrier,Major barrier,Moderate barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Somewhat likely,
20,2026-06-11 23:59:39,2026-06-12 00:04:52,anonymous,,,Other;,25 years or more,Slightly familiar,Maybe,Very unclear,Slightly,Neither,Agree,Neither,Neither,Neither,Case studies and learning more,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),Working with water to grow something nature friendly and useful (food or craft),,,,,,,,,Slightly,Slightly,Moderately,Slightly,Don’t trust it at  all,Neither likely nor unlikely,I dont come across it much.
21,2026-06-12 10:26:46,2026-06-12 10:34:06,anonymous,,,Farmer or agricultural land manager;,5–14 years,Moderately familiar,No,Somewhat clear,Don't trust it at all,Disagree,Strongly disagree,Strongly disagree,Agree,Disagree,Seeing it in action,Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Raising the water table to grow water loving plants but who knows if there are any commercially viable options or how practicalities would work,Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderately,Slightly,Trust it fully,Moderately,Moderately,Very likely,"I think for start the area assumed to be peat is vastly over estimated, apparently much of my land within an IDB area is assumed to be peat. This is untrue some will have been degraded many decades ago but a lot was never there. Raising the water table is fraught with difficulty, land need laser levelling to be flat enough to do and there is problems with holding water  up in watercourses because that can negatively impact your neighbours who may not want to carry out paludiculture. In short I think it's unproven, unrealistic with no good business case or practical examples of it working"
22,2026-06-12 13:53:25,2026-06-12 14:05:45,anonymous,,,Conservation or ecological practitioner ; NGO or third sector worker;,5–14 years,Very familiar,Maybe,Somewhat clear,Slightly,Neither,Neither,Disagree,Disagree,Neither,"Seeing real life examples, hearing from land managers doing it, understanding finances",Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),It's a way to grow crops in a wetter environment and as well as being productive it helps protect peat soils and increases resilience of the land in a changing cliamte,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Somewhat likely,"I think it depends who you're communicating with and what their understanding is. For people unfamiliar with the techniques, ""rewetting"" could make them think their land is going to be flooded. Similarly, creation of bog pools or dams could suggest images of deep pools that could be a risk to livestock. This is why it's always good to share clear info, photos etc and better still get out on site with people."
23,2026-06-12 14:17:55,2026-06-12 14:55:43,anonymous,,,Conservation or ecological practitioner ;,Less than 5 years,Slightly familiar,Yes,Somewhat clear,Don't trust it at all,Agree,Agree,Strongly disagree,Agree,Disagree,Seeing a working example and talking to someone who runs an economically viable paludiculture business.,Motivating (they point to something I care about),Motivating (they point to something I care about),Motivating (they point to something I care about),Wet-farming is about working with the land rather than against it. It makes so much more sense to lean into planting crops that suit the hydrology and geography of your land rather than putting resources into turning your land into something that hydrologically it doesn't want to be.,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Major barrier,Moderate barrier,Moderate barrier,Trust it fully,Trust it fully,Moderately,Moderately,Trust it fully,Neither likely nor unlikely,"The way it is communicated works well for securing more public and private funding for peatland restoration, I think we're good at covering the many benefits of peatlands, e.g. water quality, fire risk reduction, carbon, biodiversity etc. However, it does not work well for other land users, e.g. farmers and estate owners who feel like they have no choice in the matter - that they are being told by authorities that this is what's happening whether they like it or not. This puts farmers and other land users off doing peatland restoration on their land, and creates friction between groups. Farming communities are close knit and often suspicious of large organisations, messaging about the benefits of peatland restoration needs to come from within these communities - from the farmers that have been brave enough to give it a go, and from practitioners who are willing to go to farmers and speak to them in a more human way."
24,2026-06-12 14:53:57,2026-06-12 15:05:10,anonymous,,,Conservation or ecological practitioner ;,5–14 years,Very familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Neither,Agree,Neither,Having an experienced farmer discuss it with me in person,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"It’s not about flooding out your land or taking away your farm, it’s about working with the land to protect it. With peaty soils, when they dry out they shrink and break down, so by having the water closer to the surface we can keep those soils for longer and avoid damaging them as much as we are",Moderate barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Major barrier,Moderate barrier,Moderately,Moderately,Trust it fully,Trust it fully,Moderately,Somewhat likely,"It helps communicate the measurable outcomes (hectares, spend, water table) to the target audience for some projects, but that target audience is often grant funding bodies and other peatland restoration projects. I think that there is a lack of recognition of the need to bring communities with us from some parts of our sector and the language used which can be overly technical or pitched at the wrong level can hinder the connection people can form with the concept. With farmers specifically i think that it’s sometimes lost on restoration practitioners that farmers are not a monolith, and we are often talking to people with a deep connection to land and place, even if we may disagree on how best to manage the land. Part of the case for change needs to be a discussion of what these landscapes need with empathy for those who engage with them beyond the lifespan of our grants or projects at the heart of what we do, being able to use language and stories that reflect the people we want to work with will help everyone"
25,2026-06-12 16:22:06,2026-06-12 16:28:11,anonymous,,,Policy advisor or government officer;,Less than 5 years,Slightly familiar,Yes,Somewhat clear,Slightly,Agree,Strongly agree,Agree,Disagree,Neither,Talking to someone doing it,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Overly academic (not how I’d naturally talk about it),,Major barrier,Moderate barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Moderate barrier,Major barrier,Moderately,Moderately,Trust it fully,Moderately,Moderately,Somewhat likely,"I think we need to focus on wider benefits not just carbon benefits, this is alienating to people especially in a cost of living crisis. Climate resilience and nature restoration are much more real benefits for people. Also an acknowledgement of the cultural significance of land and being a farmer, generational practices ect. How does that work with evolving land use and how are we communicating with these groups"
26,2026-06-12 15:01:32,2026-06-12 18:44:52,anonymous,,,Farmer or agricultural land manager;,25 years or more,Very familiar,Yes,Somewhat clear,Moderately,Agree,Agree,Agree,Agree,Agree,Having examples of how barriers could be overcome to enable it to be undertaken - ie. planning and regulation barriers preventing the ability to have water storage and abstraction which are critical to a wetter farming model.,Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),Motivating (they point to something I care about),"Still being able to continue to produce the crops you have always done, or even some differnt crops you havent yet considered but just raising the water table below the surface which then also helps to reduce the emissions coming from the land.",Moderate barrier,Major barrier,Moderate barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Major barrier,Slightly,Moderately,Trust it fully,Moderately,Slightly,Very likely,Hinders the case as the current narrative is that it is all about restoration ie. removing farming from the peatland and thus livelihoods - the ability to still restore peatland but also continue to operate and farm is often lost in the narrative and so disengages the farming community from looking into it further.
27,2026-06-13 09:39:44,2026-06-13 09:52:29,anonymous,,,Researcher or academic ;,Less than 5 years,Slightly familiar,Maybe,Somewhat clear,Slightly,Strongly agree,Strongly agree,Neither,Disagree,Agree,"More visibility of products and value chains, more images and illustrations of the work in the land",Overly academic (not how I’d naturally talk about it),Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),"Growing plants like moss, reed etc that are water-loving, for which we need wet soils",Moderate barrier,Moderate barrier,Major barrier,Moderate barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Trust it fully,Moderately,Trust it fully,Moderately,Trust it fully,Very likely,"I feel like there are organisations wkth strong visual comminication. Yet the main barrier for me is availability of land. Land is used in drainage agriculture, thus introducing more narratives that entail productivity with rewetting is crucial to reach goals of rewetting landscape. In the end, most of it needs to come from agricultural areas, thus narratives of pure nature conservation without producitivity will spark repulsion. Here more images of the work, of the product, of the different steps of land work and in the production, explaining the processes of transforming materiality from land, agricultural produce, to fabrication of materials and products will be helpful."
28,2026-06-13 11:40:26,2026-06-13 11:57:46,anonymous,,,Farmer or agricultural land manager;,25 years or more,Slightly familiar,Maybe,Very unclear,Slightly,Disagree,Agree,Neither,Disagree,Neither,Seeing and hearing about examples and more importantly seeing the financials,Alienating (they don’t feel relevant to my work),Technical (useful but not emotionally engaging),Overly academic (not how I’d naturally talk about it),"I wouldn't know where to start! Somehow we've just got to make a return on the land we have got. You've got to work with the seasons, got to accept that once the water comes up, work comes to a standstill.  As for what crops to grow, really unsure about this. My experience is with grazing  livestock.",Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Minor barrier,Moderate barrier,Moderate barrier,Don’t trust it at  all,Don’t trust it at  all,Trust it fully,Slightly,Slightly,Somewhat likely,"I think the communication is focused on certain areas, and therefore the knowledge tends to stay very local. Many farmers are interested in how others farm, they also are keen to learn about other systems. If the knowledge transfer was on a more regional, even national scale, you would have more interest from those to whom it isn't even relevant.  This would allow for questioning and observations with an outside perspective. First and foremost,  we need to know how.one can make a business of managing peatlands.  Are carbon market credits the silver bullet? Should we be mknetising biodiversity and nature? Or are there crops and livestock that can genuinely work? The comms at the moment focus on restoration rather than business change. Lets see the real options available to farming businesses that may even see an opportunity to invest in peatland restoration."
29,2026-06-13 10:58:11,2026-06-13 11:58:55,anonymous,,,NGO or third sector worker;,5–14 years,Very familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Disagree,Agree,Strongly agree,"Having videos of successful paludiculture sites, interspersed with land owner commentary, seeing these sites in person, seeing the whole production line",Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"So basically its about trying to preserve the peat soils we have, theyre braking down at the minute because they've made of dead plants which once exposed to air decay really quickly, we can stop this, stop the soil loss by making the peat wet again, but understandably this has huge impacts on a farm buisness, it sounds a bit like flooding land you've spent ages draining but its more about trying to carefully raise water levels in a controlled way across targeted land, of course the larger these land parcels are the easier and cheaper it is. Naturally theres still huge development to do, most of the crops are wetland crops so that's a change in what youre growing and with that a change in machinery so for example a big one at the minute if bullrush for the seed heads for insulation and the remaining material for construction boards. This is a small market at the minute but imagine if you were growing the material to build the homes for the people around you, whilst preserving the peat for future generations of farmers all the while helping clean up water, lock in carbon and provide a beautiful nature ruch landscape. Then you really would be a really incredible steward of the countryside",Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Major barrier,Moderately,Moderately,Trust it fully,Moderately,Slightly,Very likely,"Hinders, perhaps it is too dreamy too nature focused, not portrayed with a certainty that these things need to be done for a viable livable planetary future"
30,2026-06-15 16:03:14,2026-06-15 16:08:39,anonymous,,,Conservation or ecological practitioner ;,5–14 years,Slightly familiar,No,Very unclear,Moderately,Agree,Agree,Neither,Disagree,Neither,Working example and literature around a project as it develops and then runs,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Minor barrier,Minor barrier,Moderate barrier,Minor barrier,Moderately,Trust it fully,Moderately,Trust it fully,Moderately,Neither likely nor unlikely,I feel a lot of the restoration goes towards wildlife conservation rather than also demonstrating a financial incentive to farmers and showing that it is also economically viable to farm this type of land.
31,2026-06-15 16:19:07,2026-06-15 16:24:49,anonymous,,,Conservation or ecological practitioner ;,25 years or more,Slightly familiar,No,Very unclear,Don't trust it at all,Disagree,Disagree,Disagree,Strongly disagree,Neither,Visiting a working project,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),Making use of wetland and making it productive for farming,Moderate barrier,Moderate barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Minor barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Somewhat likely,I'm not sure I have heard it communicated at all.
32,2026-06-15 16:46:18,2026-06-15 16:52:21,anonymous,,,NGO or third sector worker;,Less than 5 years,Not at all familiar,No,Very unclear,Slightly,Disagree,Agree,Agree,Strongly disagree,Agree,An example which has been up and running for several years with backed evidence that it is viable,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Overly academic (not how I’d naturally talk about it),,Major barrier,Moderate barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Slightly,Moderately,Trust it fully,Moderately,Moderately,Somewhat likely,
33,2026-06-16 11:38:40,2026-06-16 11:52:30,anonymous,,,NGO or third sector worker;,5–14 years,Very familiar,Yes,Vivid and clear,Moderately,Strongly agree,Agree,Agree,Agree,Agree,Clear evidence that the economics stack up / seeing paludiculture products on shop shelves / additional govt long term policy & funding support (in addition to the new ELMs payments for higher water table management on agri peat),Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),I wouldn't use the term 'wet-farming' as i think that would be alienating. I would say wetter farming is about raising the water table and growing crops suited to those wetter conditions; its continues productive use of the land but also protects the peat soils,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderately,Trust it fully,Trust it fully,Moderately,Moderately,Very likely,"Peatland 'restoration' is about taking land out of production, out of farming, so talking about 'restoration' to farmer would be alienating. The practise of paludiculture is not restoration so I wouldn't use the term 'restoration' in conversations about wetter farming/paludiculture with farmers (even though paludiculture can be seen as a step on a restoration trajectory to eNGOs/policy makers, and can also play a part within a landscape mosaic, for example buffering conservation areas within an drained/intensively farmed landscape)"
34,2026-06-16 12:08:33,2026-06-16 12:13:19,anonymous,,,Conservation or ecological practitioner ;,25 years or more,Not at all familiar,No,Very unclear,Slightly,Strongly disagree,Agree,Neither,Strongly disagree,Strongly disagree,seeing a working example,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Minor barrier,Moderately,Slightly,Moderately,Trust it fully,Slightly,Neither likely nor unlikely,
35,2026-06-16 13:16:20,2026-06-16 13:23:03,anonymous,,,Conservation or ecological practitioner ; NGO or third sector worker; Researcher or academic ;,5–14 years,Very familiar,Yes,Somewhat clear,Slightly,Strongly agree,Strongly agree,Agree,Strongly agree,Strongly agree,we need to develop the full business model which is resillient and not reliant from a farming perspective on grants,Motivating (they point to something I care about),Technical (useful but not emotionally engaging),Unfamiliar (I’m still getting to grips with the language),what we are looking to develop is a landscape mosaic business model which works economically and is sustainable for multiple generations.,Major barrier,Major barrier,Major barrier,Minor barrier,Minor barrier,Moderate barrier,Moderate barrier,Major barrier,Moderately,Moderately,Moderately,Slightly,Slightly,Very likely,We need to be very clear with the language used and make sure our terminology matches other stakeholders
36,2026-06-16 15:56:40,2026-06-16 16:07:50,anonymous,,,Conservation or ecological practitioner ;,5–14 years,Moderately familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Agree,Agree,Agree,"definitely visiting a working example, and studies which compare yield (crop weight and financial) across farming systems where the only difference is wet. This also needs follow-up into supply chains to see where the markets are, what we would potentially be missing, and whether our cutting emissions from peatlands and wetlands would simply be exported by diverting rather than reducing demand for certain products. I would also like more evidence on emissions data on rewetted land, so long-term experiments, and guidance on how these can be implemented by small scale farms. There should be some tool or advice available for farmers who think they have appropriate land to get it tested and modelled.",Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"raising the water table to a natural level, removing or much reducing drainage, and allowing areas to be restored to a natural(ised) hydrology. At the same time, we'd be keeping the land in productivity, but it would almost definitely mean switching products (with associated costs of changes to farming systems). So there's crops you can grow on wet land that don't mind being flooded, like typha, or some vegetables like lettuce and chinese leaf that don't mind a very high water table. Wet farming means the soil is kept in a more natural state, locks in carbon, it's a form of natural flood protection, there's lots of benefits - but initial costs can be high and it's still quite experimental in the UK.",Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Moderately,Moderately,Trust it fully,Trust it fully,Trust it fully,Somewhat likely,"I think it can be a bit obfuscated - seems like research is mostly high-level academic, and not always applicable to real-life farms. It needs a big leap of faith to translate that when capital and assets are at risk. There's also not really a safety net to try new things in farming. I think the biggest thing is seeing an example in practice. Then it feels more possible. And having data on actual markets for products (like typha - emerging market, but not enough to keep hundreds or thousands of farms in business if they switched right now). So I think that language and stories are important, but currently a bit inaccessible or limited. I've been to one, a talk by the G fresh farm. It doesn't seem widely available. I think it's also got too much of an academic, technical, experimental feel at the moment, not tangible and accessible enough to feel like a real option for most farmers."
37,2026-06-17 08:20:07,2026-06-17 08:32:14,anonymous,,,NGO or third sector worker;,5–14 years,Very familiar,Yes,Vivid and clear,Moderately,Strongly agree,Strongly agree,Strongly agree,Agree,Agree,"While a lot of work is being done across the country on proving wetter farming/paludiculture as a concept, we still do not have a complete business case to prove the method is both financially and environmentally beneficial - this is not too far away. We will not see a major switch from business as usual to wetter farming until this business case has been proved. Now that we have working trials, we can use these as demonstrator sites to show farmers and landowners what is achievable.",Motivating (they point to something I care about),Motivating (they point to something I care about),Overly academic (not how I’d naturally talk about it),"Wetter farming is taking a piece of land that you are no longer able to farm traditionally with a crop such as wheat/potatoes because the land has become too wet and work with the water to grow a crop, which as typha, that can tolerate those higher water levels. This would allow this field to become productive once again whilst the rest of the land holding can remain business as usual.",Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Major barrier,Major barrier,Moderately,Trust it fully,Trust it fully,Trust it fully,Moderately,Very likely,"Traditionally, peatland restoration has been communicated as taking a piece of degraded bog and turning it back into functioning lowland raised bog - there is no room for farming in this scenario. However, wetter farming is not peatland restoration it is about raising the water table to protect the peat soils in a way that allows farming to continue. We need to ensure that it is clear that wetter farming is an alternative farming method and we are not asking farmers to give over their land to 'unproductive' restoration. Wetter farming is not about taking profitable arable land out of production, it is about finding ways to make those areas of land that are already incredibly wet - largely due to unpredictable weather patterns - and allow them to be farmed once again, using a new method."
38,2026-06-17 10:14:13,2026-06-17 10:22:43,anonymous,,,Conservation or ecological practitioner ;,Less than 5 years,Not at all familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Neither,Disagree,Agree,seeing a working example,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),Working with re-wetted land to grow produce that thrives in that environment.,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Minor barrier,Major barrier,Major barrier,Moderate barrier,Moderately,Trust it fully,Moderately,Trust it fully,Moderately,Neither likely nor unlikely,"At the moment, peatland restoration is often depicted as serving wildlife and nature, not paludiculture. They could be interpreted as being at odds with one another."
39,2026-06-17 12:43:48,2026-06-17 12:57:58,anonymous,,,NGO or third sector worker;Conservation or ecological practitioner ;,5–14 years,Not at all familiar,No,No picture at all,Don't trust it at all,Disagree,Agree,Disagree,Strongly disagree,Agree,All the above. I have never heard of it really before this questionnaire. I would need to hear about it as a possibility.,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),Something to do with growing wet crops..? Perhaps like asian rice paddies?,,,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Major barrier,,Moderately,Trust it fully,Moderately,Trust it fully,Moderately,Somewhat unlikely,"I would say the VAST majority don't know how important peatlands are. This is not only poorly communicated, I would say it is not communicated. Other than talking to those already working in ecology or conservation, whenever I have mentioned the importance of peatland people have without fail had no idea. A lot don't even know what peat is. I had never heard of Paludiculture before this survey, still not really sure what it is. But I will google it after this and try to inform myself."
40,2026-06-15 10:35:19,2026-06-18 14:08:20,anonymous,,,Researcher or academic ;,5–14 years,Moderately familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Neither,Strongly agree,Agree,"Probably the financial argument is what is going to convince most people and having a market for the products, supply chains etc.",Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),It's like growing a crop without having to drain the land because the plants like wet soil.,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Moderate barrier,Trust it fully,Trust it fully,Trust it fully,Trust it fully,Trust it fully,Very likely,"I feel that lots of people are doing a lot of good work but there is always room for better communication, especially in academia where i work, but also more widely, across govt. departments, local authorities and sharing knowledge with land managers, farmers and communities. I don't think it hinders change how it is currently being done, but there is a need for more comms, and actually engaging experts in marketing and pr, rather than thinking we can do it all. Organisations spend lots of money on restoration projects, but rarely is there a budget for comms, social media and marketing, to my knowledge anyway!"
41,2026-06-24 15:55:43,2026-06-24 16:26:57,anonymous,,,NGO or third sector worker;,Less than 5 years,Very familiar,Yes,Somewhat clear,Moderately,Agree,Agree,Neither,Disagree,Disagree,"Real-world examples of practicing farms managing paludiculture, with a clear supply chain and market for goods.",Motivating (they point to something I care about),Motivating (they point to something I care about),Motivating (they point to something I care about),"Wet farming is a way of growing crops or raising livestock in areas of land which would naturally be wet. In artificially drained places, it involves raising the water table by blocking drains and managing water in a different way. Paludiculture (true wet farming) can be used to grow crops which are adapted to growing in wet conditions (like blueberries), crops for building materials (like reeds) and even livestock like water buffalo. Other forms of wet farming may only raise the water table a bit, so other crops can be grown while preventing some loss of carbon from the soil. This isn't as good for the environment as true paludiculture, but can be really helpful in learning how to work with water when farming.",Major barrier,Major barrier,Moderate barrier,Moderate barrier,Minor barrier,Minor barrier,Moderate barrier,Not a barrier,Moderately,Moderately,Moderately,Moderately,Slightly,Very likely,"There are some great examples of communication about peatland restoration which clearly outline the multiple benefits it brings for people and nature, but there can sometimes be too much of a focus on carbon benefits. Overly scientific or technical language is also a barrier to reaching diverse audiences, but lots of eNGOs (like the Wildlife Trusts, National Trust and RSPB for example) are great at communicating in accessible ways. The challenge for them is that they are often preaching to the choir. Creative approaches to engaging people using storytelling, connections to cultural heritage, artwork and participation are critical to making peatland restoration more accessible. Examples like the case study films by Peatland ACTION and South West Peatland Partnership's 'The Living Layer' and Art and Energy's 'Bog Casts' bring a human voice to peatland restoration - local, place-based examples are what people can relate to."
42,2026-06-25 09:04:34,2026-06-25 09:16:24,anonymous,,,Conservation or ecological practitioner ;,Less than 5 years,Very familiar,Yes,Somewhat clear,Slightly,Agree,Agree,Neither,Agree,Agree,Working examples of farms implementing paludiculture practices outside of any 'special' funding for paludiculture eg trials.,Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"Raisning the water level in the soil to be closer to the surface, and finding crops/livestock that are better suited to these conditions. Blocking and breaking the drains that dry out the soil.",Major barrier,Minor barrier,Major barrier,Major barrier,Minor barrier,Moderate barrier,Moderate barrier,Minor barrier,Trust it fully,Moderately,Trust it fully,Trust it fully,Trust it fully,Somewhat likely,"I think that the stories don't hinder change at the moment. I think if these stories haven't developed to show farms having established paludiculture practices in the coming years, they may start to become a barrier. Realistically, I don't believe farms will be willing to invest in, and take the risk of, changing to paludicutural practices unless they can see farms that have already made the transition with obvious financial success that repays the investment."
43,2026-06-25 09:16:09,2026-06-25 09:35:44,anonymous,,,Policy advisor or government officer;,25 years or more,Very familiar,Yes,Very unclear,Moderately,Agree,Agree,Agree,Agree,Agree,More demonstration sites,Motivating (they point to something I care about),Technical (useful but not emotionally engaging),Motivating (they point to something I care about),"Wet farming is farming on land that has had the water table raised so that less carbon is being released to the atmosphere. Every 10 cm of water table raise reduces the carbon released significantly and helps to stop the peat soils from degrading and eroding. its best to investigate what changes would work best for your farming business economically and practically. These could be continuing with your usual crops or grazing with a raise in water table, growing different crops such as Typha, using an CSHT scheme or restoration for bodiversity.",Major barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Trust it fully,Trust it fully,Trust it fully,Moderately,Moderately,Neither likely nor unlikely,"I think there is an issue with confusion on the use of the term restoration now that we are looking to reduce carbon emissions from lowland agricultural peat, as restoration in this concept is sometimes used to mean raising water tables to protect peat soils rather than restoring as in biodiversity habitats."
44,2026-06-25 09:42:51,2026-06-25 09:59:06,anonymous,,,Conservation or ecological practitioner ;,5–14 years,Moderately familiar,Yes,Somewhat clear,Slightly,Neither,Agree,Neither,Neither,Agree,More examples of paludiculture in action in Ireland; being able to see it in action within the climate we have here and talking to practitioners about it,Motivating (they point to something I care about),Motivating (they point to something I care about),Technical (useful but not emotionally engaging),"Wet farming is working with the natural movement of water in the landscape, rather than draining the land to be productive",Moderate barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Minor barrier,Major barrier,Moderate barrier,Slightly,Trust it fully,Trust it fully,Moderately,Moderately,Somewhat likely,"I feel that the way restoration is communicated can be a major help for change; however, we need to be aware that not everyone will fall under the conservationist or 'nature-lover' category, and we need to be able to reach a wider range of people for a wider range of reasons. many working class people feel the impacts of climate change and a lack of access to green spaces, or the ecological services they provide (e.g. things like living in flood zones), but due to sociological issues, can often feel disconnected from these spaces. Wealth inequality causes people to have different priorities, and it can often be a privilege to focus your attention on causes that are beyond your basic needs. Farmers are also often stewards of the land, but can feel alienated by the language we use, and feel like the enemy of conservation, when getting them on board and working together would be far more successful. I think that working towards our end goal (which could be a world where nature is thriving and protected) means we need to focus on stories and language which engages a lot of different people in a lot of different ways. We need to think about those who would benefit the most, rather than fall into an echo chamber of people who are already 'believers'"
45,2026-06-25 12:09:21,2026-06-25 12:22:27,anonymous,,,Researcher or academic ;,25 years or more,Very familiar,Maybe,Somewhat clear,Slightly,Neither,Agree,Neither,Agree,Agree,Growing short rotation willow on reclaimed/rewetted peat OK until harvest - OK in Scandinavia where wetland freezes in winter allowing harvest.,Technical (useful but not emotionally engaging),Motivating (they point to something I care about),Overly academic (not how I’d naturally talk about it),Come back in a couple of years and I will show you how it has worked (or not!),Major barrier,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Moderately,Slightly,Slightly,Slightly,Don’t trust it at  all,Somewhat unlikely,"I'm pleased with the way peatland restoration has reached the public consciousness particularly with CO2 capture and peat free composts. However many peat-free composts, for example are rubbish and gardeners are having bad experiences with them - need to be confident of our answers before we present them to the public including farmers."
46,2026-06-25 18:11:52,2026-06-25 18:15:55,anonymous,,,Conservation or ecological practitioner ;,25 years or more,Moderately familiar,Yes,Somewhat clear,Moderately,Disagree,Strongly agree,Agree,Disagree,Disagree,modelliong,Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),Technical (useful but not emotionally engaging),its about working with the bog not against it,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Minor barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Very likely,need more variety of examples
47,2026-06-27 07:47:05,2026-06-27 07:52:59,anonymous,,,Farmer or agricultural land manager;,25 years or more,Not at all familiar,No,Somewhat clear,Don't trust it at all,Neither,Strongly agree,Strongly disagree,Neither,Strongly disagree,Returns per hectare $ £,Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),?,Major barrier,Major barrier,Minor barrier,Moderate barrier,Minor barrier,Major barrier,Moderate barrier,Major barrier,Don’t trust it at  all,Slightly,Slightly,Don’t trust it at  all,Don’t trust it at  all,Somewhat unlikely,I farm to grow food simple really !!!!!
48,2026-06-27 10:17:49,2026-06-27 10:26:07,anonymous,,,Farmer or agricultural land manager;,25 years or more,Not at all familiar,No,Very unclear,Don't trust it at all,Strongly disagree,Strongly disagree,Disagree,Strongly agree,Neither,Seeing it working,Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Alienating (they don’t feel relevant to my work),Destroying the food producing capabilities of good agricultural land to salve the conscience of academics,Major barrier,Major barrier,Major barrier,Major barrier,Minor barrier,Major barrier,Major barrier,Major barrier,Slightly,Don’t trust it at  all,Moderately,Don’t trust it at  all,Don’t trust it at  all,Somewhat unlikely,"It helps the academic case, but completely alienates farmers who see the massive efforts of farmers over hundreds of years being cast aside while those that call for it have no skin in the game and are insulated from the consequences."
49,2026-06-28 09:19:02,2026-06-28 09:27:36,anonymous,,,Conservation or ecological practitioner ;Farmer or agricultural land manager;,5–14 years,Not at all familiar,No,Very unclear,Don't trust it at all,Disagree,Agree,Strongly disagree,Neither,Strongly disagree,Seeing a working example is always best imo,Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),I do not feel I have enough knowledge to discuss the topic in order to explain it to another as I work on dry downland sites,Moderate barrier,Moderate barrier,Major barrier,Major barrier,Minor barrier,Minor barrier,Major barrier,Minor barrier,Slightly,Moderately,Trust it fully,Trust it fully,Moderately,Neither likely nor unlikely,"I think it is a mixed bag an depends entirely upon your audience, I see mostly conservation based projects, but then that is my area of interest. Agriculturally I have seen very little if any that are based on viability of industry whilst carrying out restoration, again that could well be my sphere that I exist in. For me it helps change, not that I have any wetland sites, but it is great to see sites being restored, if I were in agriculture for financial survival or profit then so far I would struggle to see how it applied to my model of operation, but again that could be a lack of me seeing such focused projects."
50,2026-06-29 10:08:10,2026-06-29 10:13:50,anonymous,,,Conservation or ecological practitioner ;,15–24 years,Slightly familiar,Maybe,Somewhat clear,Slightly,Disagree,Agree,Neither,Strongly disagree,Neither,"All of the above, I'd simply need to invest time into understanding the science, finance, etc. whether that is via reports, working examples or meeting people involved in the industry",Motivating (they point to something I care about),Motivating (they point to something I care about),Unfamiliar (I’m still getting to grips with the language),"There's other ways to make a living off the bog that others are at, looking at basically switching from sheep to growing crops. It's a new thing alright but sure there's no money in sheep anymore anyways",Moderate barrier,Moderate barrier,Major barrier,Major barrier,Major barrier,Major barrier,Major barrier,Moderate barrier,Moderately,Moderately,Moderately,Moderately,Moderately,Neither likely nor unlikely,"In an Irish context, the wording matters. Terms such as rewetting are alienating, and are best avoided in place or other terms such as drain management, or revegetating."
51,2026-06-30 14:59:17,2026-06-30 15:14:32,anonymous,,,Farmer or agricultural land manager;,5–14 years,Very familiar,Maybe,Very unclear,Don't trust it at all,Neither,Strongly disagree,Strongly disagree,Agree,Neither,There is a lot about environmental benefits but very little agronomic and economic detail to suggest it is a viable way to farm.,Alienating (they don’t feel relevant to my work),Motivating (they point to something I care about),Overly academic (not how I’d naturally talk about it),Trying to farm on black fen with a really high water table (usually met with a response of 'how are you meant to grow anything?'),Major barrier,Major barrier,Moderate barrier,Moderate barrier,Minor barrier,Moderate barrier,Minor barrier,Major barrier,Don’t trust it at  all,Slightly,Moderately,Moderately,Moderately,Neither likely nor unlikely,"Peatland restoration is completely different to paludiculture. Restoration implies reverting peatland back to a near natural state with the intention of regenerating peat. Paludiculture is about slowing degradation of productive lowland agricultural peat soils. Coming from a farming background and having briefly worked in the world of paludiculture, it feels like it has been completely owned by NGOs and academics who have used it as an opportunity to access copious grants from the DEFRA pot to show emissions reductions, which doesn't do any favours for its perception by farmers."
52,2026-07-02 19:35:01,2026-07-02 19:37:45,anonymous,,,Farmer or agricultural land manager;,25 years or more,Slightly familiar,No,Very unclear,Slightly,Neither,Disagree,Neither,Disagree,Neither,,Unfamiliar (I’m still getting to grips with the language),Technical (useful but not emotionally engaging),Unfamiliar (I’m still getting to grips with the language),,Major barrier,Major barrier,Moderate barrier,Moderate barrier,Moderate barrier,Major barrier,Moderate barrier,Moderate barrier,Don’t trust it at  all,Slightly,Moderately,Slightly,Slightly,Neither likely nor unlikely,
"""

df = pd.read_csv(io.StringIO(survey_csv))
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df

## 1. Sample overview

In [ ]:
role_col = df.columns[6]
experience_col = df.columns[7]

df['is_farmer'] = df[role_col].astype(str).str.contains('Farmer or agricultural land manager', na=False)

print("n =", len(df))
print("Farmers / land managers:", df['is_farmer'].sum())
print("Rest of sample:", (~df['is_farmer']).sum())

fig, ax = plt.subplots(figsize=(7,4))
df[experience_col].astype(str).str.strip().value_counts().plot(kind='barh', ax=ax, color='#3b6e5e')
ax.set_title('Years worked with or managed land')
ax.set_xlabel('Respondents')
plt.tight_layout()
plt.show()

In [ ]:
import re

clarity_col = df.columns[10]        # Q10: "how clear is that picture?" (single-select, 4-point)
trust_col = df.columns[11]          # Q11: economic-trust item (4-point) — used throughout the thesis
picture_agree_col = df.columns[12]  # Q12: "I can picture a thriving landscape..." (5-point agreement) — the item Methodology §3.2 specifies for the clarity construct

clarity_map = {'No picture at all': 1, 'Very unclear': 2, 'Somewhat clear': 3, 'Vivid and clear': 4}
trust_map = {"Don\'t trust it at all": 1, 'Slightly': 2, 'Moderately': 3, 'Trust it fully': 4}
agree_map = {'Strongly disagree': 1, 'Disagree': 2, 'Neither': 3, 'Agree': 4, 'Strongly agree': 5}

def clean(s):
    """Normalises two things the MS Forms export does inconsistently across
    columns: a trailing non-breaking space (\xa0) on almost every cell, and a
    curly apostrophe (\u2019) in some columns (e.g. the messenger-trust items)
    where others use a straight one (\'). Without this, "don\'t trust it at
    all" responses in the affected columns would be missed by any exact-match
    lookup."""
    if pd.isna(s):
        return s
    s = str(s).replace("\xa0", " ").replace("\u2019", "\'")
    return re.sub(r"\s+", " ", s).strip()

df['clarity_raw'] = df[clarity_col].apply(clean)
df['trust_raw'] = df[trust_col].apply(clean)
df['picture_agree_raw'] = df[picture_agree_col].apply(clean)

df['clarity_score'] = df['clarity_raw'].map(clarity_map)
df['trust_score'] = df['trust_raw'].map(trust_map)
df['picture_agree_score'] = df['picture_agree_raw'].map(agree_map)

# Exact-match boolean coding per thesis Methodology §3.2:
# clarity is "closed" (clear) for Agree/Strongly agree on Q12; "neither" is coded not-clear.
# trust is "closed" for Moderately/Trust it fully on Q11.
df['clear'] = df['picture_agree_score'] >= 4
df['trust_open'] = df['trust_score'] >= 3

df[['clarity_raw','clarity_score','trust_raw','trust_score','picture_agree_raw','picture_agree_score','clear','trust_open']].head()


## 3. The imagination gap / trust gap matrix (Figure 2)

Scored exactly as specified in the thesis's Methodology (S3.2): imaginative
clarity from **Q12** ("I can picture what a thriving paludiculture landscape
would look like"), coded clear for Agree/Strongly agree and not-clear
otherwise (including "Neither"); economic trust from **Q11**, coded trusting
for Moderately/Trust it fully. This is a fixed-anchor coding, not a median
split — matching the thesis exactly reproduces its reported n's (26/13/11/2),
which a median split will not.

In [ ]:
def quadrant(row):
    if pd.isna(row['clear']) or pd.isna(row['trust_open']):
        return np.nan
    if row['clear'] and row['trust_open']:       return 'Sees it, trusts it'
    if not row['clear'] and row['trust_open']:   return 'Cannot see it, trusts it'
    if row['clear'] and not row['trust_open']:   return 'Sees it, does not trust it'
    return 'Cannot see it, does not trust it'

df['quadrant'] = df.apply(quadrant, axis=1)
counts = df['quadrant'].value_counts()
n_total = counts.sum()
pct = (counts / n_total * 100).round(0).astype(int)

comparison = pd.DataFrame({
    'n': counts,
    '%': pct,
    'Thesis-reported %': pd.Series({'Cannot see it, does not trust it': 50, 'Sees it, trusts it': 25,
                                     'Sees it, does not trust it': 21, 'Cannot see it, trusts it': 4})
})
display(comparison)

n_bl = counts.get('Cannot see it, does not trust it', 0)
n_tr = counts.get('Sees it, trusts it', 0)
n_br = counts.get('Sees it, does not trust it', 0)
n_tl = counts.get('Cannot see it, trusts it', 0)

fig, ax = plt.subplots(figsize=(7.2, 6.2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axhline(0.5, color='black', lw=1)
ax.axvline(0.5, color='black', lw=1)

quad_boxes = [
    (0, 0.5, 0, 0.5, '#f2b6ae', f'Cannot see it,\ndoes not trust it\nn = {n_bl} ({100*n_bl//n_total}%)', 'Neither gap closed'),
    (0.5, 1, 0, 0.5, '#f7cf8a', f'Sees it,\ndoes not trust it\nn = {n_br} ({100*n_br//n_total}%)', 'Can picture the landscape,\nbut not its economic viability'),
    (0, 0.5, 0.5, 1, '#a7c7e7', f'Cannot see it,\ntrusts it\nn = {n_tl} ({100*n_tl//n_total}%)', 'Convinced in principle,\nunable to picture it concretely'),
    (0.5, 1, 0.5, 1, '#a8d5ac', f'Sees it,\ntrusts it\nn = {n_tr} ({100*n_tr//n_total}%)', 'Both barriers overcome'),
]
for x0, x1, y0, y1, color, label, sub_lbl in quad_boxes:
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=color, edgecolor='none'))
    ax.text((x0+x1)/2, (y0+y1)/2 + 0.06, label, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text((x0+x1)/2, (y0+y1)/2 - 0.12, sub_lbl, ha='center', va='center', fontsize=9, style='italic', color='#333333')

ax.set_xticks([0, 1]); ax.set_xticklabels(['Low', 'High'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Low', 'High'])
ax.set_xlabel('Economic trust  \u2192', fontsize=11)
ax.set_ylabel('Imaginative clarity  \u2192', fontsize=11)
ax.set_title('The Imagination Gap / Trust Gap Matrix', fontsize=14, fontweight='bold', pad=15)
fig.text(0.5, 0.02,
    f'Stakeholder survey, n = {n_total}. Imaginative clarity: Likert item (Q12), "neither agree nor disagree" coded as not-clear.\n'
    'Economic trust: single-select item (Q11), "Moderately"/"Trust it fully" coded as trust.',
    ha='center', fontsize=7.5, color='#444444')
fig.text(0.5, -0.01, f'Figure 2 The imagination gap / trust gap matrix, cross-tabulating imaginative clarity\nagainst economic trust (stakeholder survey, n = {n_total}).',
    ha='center', fontsize=8)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('figure_2_imagination_trust_matrix.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
from scipy import stats

CLARITY_ORDER = [1, 2, 3, 4, 5]
CLARITY_LABELS = ['Strongly\ndisagree', 'Disagree', 'Neither', 'Agree', 'Strongly\nagree']
TRUST_ORDER = [1, 2, 3, 4]
TRUST_LABELS = ["Don\'t trust\nit at all", 'Slightly', 'Moderately', 'Trust it\nfully']

farmers = df[df['is_farmer']]
rest = df[~df['is_farmer']]
n_farmers, n_rest = len(farmers), len(rest)

u_fc, p_fc = stats.mannwhitneyu(farmers['picture_agree_score'].dropna(), rest['picture_agree_score'].dropna())
u_ft, p_ft = stats.mannwhitneyu(farmers['trust_score'].dropna(), rest['trust_score'].dropna())
print(f'Clarity (Q12) — farmers vs rest: U = {u_fc}, p = {p_fc:.3f}  [thesis: U=148.5, p=.041]')
print(f'Trust (Q11)   — farmers vs rest: U = {u_ft}, p = {p_ft:.3f}  [thesis: U=118.0, p=.005]')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def grouped_bar(ax, score_col, order, labels, title, pval):
    f_pct = farmers[score_col].value_counts(normalize=True).reindex(order).fillna(0) * 100
    r_pct = rest[score_col].value_counts(normalize=True).reindex(order).fillna(0) * 100
    x = range(len(order))
    w = 0.38
    ax.bar([i - w/2 for i in x], f_pct.values, width=w, color='#c0504d', label=f'Farmers (n={n_farmers})')
    ax.bar([i + w/2 for i in x], r_pct.values, width=w, color='#77933c', label=f'Rest of sample (n={n_rest})')
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('% within group')
    ax.set_title(title, fontsize=10.5, fontweight='bold')
    ax.text(0.98, 0.95, f'Mann-Whitney U, p = {pval:.3f}'.replace('0.', '.'),
            transform=ax.transAxes, ha='right', va='top', fontsize=9, style='italic')
    ax.legend(fontsize=8, loc='upper left')

grouped_bar(axes[0], 'picture_agree_score', CLARITY_ORDER, CLARITY_LABELS,
            '"I can picture what a thriving\npaludiculture landscape would look like" (Q12)', p_fc)
grouped_bar(axes[1], 'trust_score', TRUST_ORDER, TRUST_LABELS,
            '"How much do you trust that paludiculture\ncould be economically viable..." (Q11)', p_ft)
fig.suptitle('The messenger gap: distribution of responses, farmers vs. rest of sample', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('figure_3_messenger_gap.png', dpi=200, bbox_inches='tight')
plt.show()


## 5. Messenger trust vs. trust in economic viability itself (Figure 7)

From the raw messenger-trust items (columns 30–34) and the economic-trust item
(Q11), with the Friedman test and paired Wilcoxon significance brackets
reported in S4.3 (full tests in Section 7).

In [ ]:
messenger_cols = {
    df.columns[30]: 'Government/\nDEFRA',
    df.columns[31]: 'Conservation\nNGOs',
    df.columns[32]: 'Farmers/\nland managers doing it',
    df.columns[33]: 'Academic\nresearch',
    df.columns[34]: 'Industry bodies/\nland agencies',
}

def pct_trusting(col):
    vals = df[col].apply(clean)
    return round(100 * vals.isin(['Moderately', 'Trust it fully']).sum() / vals.notna().sum(), 0)

trust_pct = {label: pct_trusting(col) for col, label in messenger_cols.items()}
trust_pct['Economic\nviability itself'] = pct_trusting(trust_col)

ordered_labels = sorted([k for k in trust_pct if k != 'Economic\nviability itself'], key=lambda k: -trust_pct[k])
ordered_labels.append('Economic\nviability itself')

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ['#4a7c3f' if lbl != 'Economic\nviability itself' else '#c0504d' for lbl in ordered_labels]
bars = ax.bar(ordered_labels, [trust_pct[l] for l in ordered_labels], color=colors)
for b_, lbl in zip(bars, ordered_labels):
    ax.text(b_.get_x() + b_.get_width()/2, b_.get_height() + 1.5, f'{trust_pct[lbl]:.0f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('% trusting moderately or fully')
ax.set_ylim(0, 110)
ax.set_title('Trust in messengers vs. trust in the economics itself', fontsize=13, fontweight='bold')

def bracket(ax, x1, x2, y, text):
    ax.plot([x1, x1, x2, x2], [y, y+2, y+2, y], color='black', lw=1)
    ax.text((x1+x2)/2, y+3, text, ha='center', fontsize=9)

n_bars = len(ordered_labels)
bracket(ax, 0, n_bars-1, 100, '*** p < .001')
bracket(ax, ordered_labels.index('Government/\nDEFRA'), n_bars-1, 88, '*** p < .001 (DEFRA vs economic trust)')
fig.text(0.5, -0.02,
    f'Stakeholder survey, n = {len(df)}. Messenger-trust and economic-trust items share the same four-point scale.\n'
    'Even the least-trusted messenger (DEFRA) is trusted significantly more than the economics of paludiculture itself.',
    ha='center', fontsize=8, color='#444444')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('figure_4_messenger_vs_economic_trust.png', dpi=200, bbox_inches='tight')
plt.show()

pd.Series(trust_pct).sort_values(ascending=False)


## 6. Statistical tests full thesis verification

Every inferential statistic cited anywhere in the thesis, recomputed directly
from the raw responses using the exact items and coding rules in Methodology
S3.2. Thesis-reported values are shown alongside each result. Covers S4.1,
S4.2.1, S4.2.2, S4.2.4, and S4.3 — see the note near the end of this section
for why S4.2.3 and S4.5 fall outside what this notebook can mechanically
re-derive.

In [ ]:
from scipy.stats import binomtest

# S4.1: Fisher's exact on the quadrant (clarity x trust), and Spearman
ct_quad = pd.crosstab(df['clear'], df['trust_open'])
odds, p_fisher_quad = stats.fisher_exact(ct_quad)
print(f"Fisher\'s exact (clarity x trust, Q12 x Q11): OR = {odds:.1f}, p = {p_fisher_quad:.4g}  [thesis: OR=15.4, p<.001]")

sub = df.dropna(subset=['picture_agree_score', 'trust_score'])
rho, p_rho = stats.spearmanr(sub['picture_agree_score'], sub['trust_score'])
print(f"Spearman (Q12 x Q11): rho = {rho:.2f}, p = {p_rho:.2e}, n = {len(sub)}  [thesis: rho=.55, p<.001]")

# S4.2.2: barrier percentages, McNemar, and the picturing-barrier Fisher's exact
print()
barrier_cols_idx = {22: 'Financial incentive', 24: 'Technical knowledge', 28: 'Cultural unfamiliarity',
                     23: 'Policy/regulatory uncertainty', 25: 'Perception unproductive',
                     27: 'Lack of trust proven', 29: 'Tenure/ownership', 26: 'Picturing barrier'}
for idx, label in barrier_cols_idx.items():
    col = df.columns[idx]
    vals = df[col].apply(clean)
    n_valid = vals.notna().sum()
    n_major = (vals == 'Major barrier').sum()
    print(f"  {label}: {100*n_major/n_valid:.0f}% major barrier (n={n_major}/{n_valid})")

picture_barrier_col = df.columns[26]
df['picturing_major'] = df[picture_barrier_col].apply(clean).apply(
    lambda x: True if x == 'Major barrier' else (False if pd.notna(x) else pd.NA)
)
mc_sub = df[['clear', 'picturing_major']].copy()
mc_sub['imaginative_deficit'] = ~mc_sub['clear']
mc_sub = mc_sub.dropna(subset=['imaginative_deficit', 'picturing_major'])
ct_mc = pd.crosstab(mc_sub['imaginative_deficit'], mc_sub['picturing_major'])
b, c_ = ct_mc.loc[True, False], ct_mc.loc[False, True]
p_mcnemar = binomtest(min(b, c_), b + c_, 0.5).pvalue
print(f"\nMcNemar\'s exact test (imaginative deficit vs named picturing barrier): p = {p_mcnemar:.3f}  [thesis: exact p = .052]")

# Fisher\'s exact, one-tailed (the thesis\'s claim is directional: farmers "significantly
# less likely" to name the picturing barrier as major), with the 2 respondents who skipped
# this item excluded listwise — consistent with how the barrier percentages above are calculated.
ct_pb = pd.crosstab(df['is_farmer'], df['picturing_major'])
_, p_fisher_pb = stats.fisher_exact(ct_pb, alternative='less')
print(f"\nFisher\'s exact, one-tailed (farmers less likely to name picturing barrier major): p = {p_fisher_pb:.3f}")
print("  Thesis currently reads p = .039. Recomputed with the 2 non-respondents excluded")
print("  listwise (consistent with the barrier percentages above), this is p = .030.")
print("  CORRECTION NEEDED IN THESIS: p = .039 -> p = .030. Direction and significance unchanged.")


In [ ]:
# S4.2.1: already computed and plotted in Section 4 above (with Q12/Q11)
print(f"Clarity (Q12) — farmers vs rest: U = {u_fc}, p = {p_fc:.3f}  [thesis: U=148.5, p=.041]")
print(f"Trust (Q11)   — farmers vs rest: U = {u_ft}, p = {p_ft:.3f}  [thesis: U=118.0, p=.005]")

cons = df[df[df.columns[6]].astype(str).str.contains('Conservation or ecological practitioner', na=False)]
rest_c = df[~df[df.columns[6]].astype(str).str.contains('Conservation or ecological practitioner', na=False)]
u_cc, p_cc = stats.mannwhitneyu(cons['picture_agree_score'].dropna(), rest_c['picture_agree_score'].dropna())
u_ct, p_ct = stats.mannwhitneyu(cons['trust_score'].dropna(), rest_c['trust_score'].dropna())
print(f"\nConservation (n={len(cons)}) vs rest — clarity: p = {p_cc:.2f}  [thesis: p=.47]")
print(f"Conservation (n={len(cons)}) vs rest — trust:   p = {p_ct:.2f}  [thesis: p=.34]")


S4.2.4 the word itself is a hindrance (terminology register)

In [ ]:
# S4.2.4: terminology register
rewetting_col, restoration_col, paludiculture_col = df.columns[18], df.columns[19], df.columns[20]

palu = df[paludiculture_col].apply(clean)
n_neg = palu.isin(["Technical (useful but not emotionally engaging)",
                    "Unfamiliar (I\'m still getting to grips with the language)",
                    "Overly academic (not how I\'d naturally talk about it)"]).sum()
n_mot = (palu == 'Motivating (they point to something I care about)').sum()
print(f"paludiculture: technical/unfamiliar/overly academic = {n_neg}/52 ({100*n_neg/52:.0f}%)  [thesis: 40/52, 77%]")
print(f"paludiculture: motivating = {n_mot}/52 ({100*n_mot/52:.0f}%)  [thesis: 7/52, 13%]")

restor = df[restoration_col].apply(clean)
n_mot_r = (restor == 'Motivating (they point to something I care about)').sum()
print(f"peatland restoration: motivating = {n_mot_r}/52 ({100*n_mot_r/52:.0f}%)  [thesis: 35/52, 67%]")

rewet = df[rewetting_col].apply(clean)
n_mot_w = (rewet == 'Motivating (they point to something I care about)').sum()
print(f"rewetting: motivating = {n_mot_w}/52 ({100*n_mot_w/52:.0f}%)  [thesis: half the sample, 50%]")


### S4.3 Trust the messenger, not the economics

Friedman test across the five messenger-trust items, paired Wilcoxon (Farmers vs. each other messenger), and economic-viability trust vs. the scale midpoint and vs. each messenger.

In [ ]:
messenger_labels = ['Farmers', 'Academic', 'NGOs', 'DEFRA', 'Industry']
messenger_idx = {'Farmers': 32, 'Academic': 33, 'NGOs': 31, 'DEFRA': 30, 'Industry': 34}
for label, idx in messenger_idx.items():
    df[f'{label}_ord'] = df[df.columns[idx]].apply(clean).map(trust_map)
df['econ_ord'] = df['trust_raw'].map(trust_map)

mat = df[[f'{l}_ord' for l in messenger_labels]].dropna()
fr_stat, fr_p = stats.friedmanchisquare(*[mat[c] for c in mat.columns])
print(f"Friedman: chi2 = {fr_stat:.1f}, p = {fr_p:.3g}, n = {len(mat)}  [thesis: chi2=47.9, p<.001]")

print("\n-- Farmers vs each other messenger (paired Wilcoxon) --")
for label in ['DEFRA', 'NGOs', 'Academic', 'Industry']:
    ps = df[['Farmers_ord', f'{label}_ord']].dropna()
    w, p = stats.wilcoxon(ps['Farmers_ord'], ps[f'{label}_ord'], zero_method='pratt')
    print(f"Farmers vs {label}: n={len(ps)}, p = {p:.4g}")

print("\n-- Economic viability trust --")
econ_mean_03 = df['econ_ord'].mean() - 1
print(f"Mean (0-3 scale, matching thesis reporting): {econ_mean_03:.2f}  [thesis: M=1.06]")
diffs = df['econ_ord'].dropna() - 2.5
w, p = stats.wilcoxon(diffs)
print(f"vs scale midpoint: p = {p:.4g}  [thesis: p<.001]")

print("\n-- Economic viability vs each individual messenger --")
for label in messenger_labels:
    ps = df[[f'{label}_ord', 'econ_ord']].dropna()
    w, p = stats.wilcoxon(ps[f'{label}_ord'], ps['econ_ord'], zero_method='pratt')
    print(f"{label} vs Economic viability: p = {p:.4g}  [thesis: p<.001 for all, incl. DEFRA]")


### S4.2.3 and S4.5 — not mechanically verifiable

**S4.2.3** ("Plural once pictured") reports qualitative workshop artefacts — the photographs and drawings in Section 8 below — not survey statistics; nothing to recompute.

**S4.5** claims 34 of 47 respondents (72%) named "seeing a working example" as their primary answer to an open-text question. That's a manual coding judgement on free text, not a value in a column. The valid response count (47) checks out exactly; a rough keyword search independently finds 29/47 (62%), in the same range, with the gap explained by paraphrased responses a keyword match won't catch.

In [ ]:
help_col = [c for c in df.columns if c.startswith('What, if anything, would help')][0]
vals = df[help_col].apply(clean)
n_valid = vals.notna().sum()
kw = re.compile(r"working example|see it|seeing|visit|demonstrat|case stud|site visit|working project|pilot|working farm", re.I)
matches = vals.dropna().apply(lambda x: bool(kw.search(x)))
print(f"n valid responses: {n_valid}  [thesis: 47]")
print(f"Keyword sanity-check: {matches.sum()}/{n_valid} ({100*matches.sum()/n_valid:.0f}%)  [thesis: 34/47, 72%, by manual coding]")


### Summary — full-thesis audit result

Every inferential statistic in the thesis was checked against the raw survey

Two respondents skipped that barrier-ranking question. The barrier percentages
elsewhere in S4.2.2 correctly exclude them. p = .039 only reproduces if those
two are instead coded "not major" rather than excluded — inconsistent with
the rest of the section. Excluding them listwise gives p = .030 (one-tailed).


**Everything else in the thesis is correct as written and reproduces exactly:**

- OR = 15.4 (p < .001) and ρ = .55 (p < .001) for the clarity/trust
  relationship, using Q12 and Q11 as specified in Methodology §3.2.
- Both Mann-Whitney U tests for farmers vs. rest (U = 148.5, p = .041 for
  clarity; U = 118.0, p = .005 for trust), and the null results for
  conservation/ecological practitioners (p = .47, p = .34).
- All eight barrier percentages and the McNemar's exact test (p = .052).
- All four terminology-register percentages in §4.2.4.
- The Friedman test across the five messenger-trust items (χ² = 47.9,
  p < .001), every pairwise Farmers-vs-other-messenger comparison, the mean
  economic-trust score (M = 1.06 on the 0–3 scale used in the thesis), and
  every messenger-vs-economic-viability paired comparison (all p < .001).

S4.2.3 and S4.5 rest on qualitative workshop artefacts and open-text coding,
— nothing there contradicts the data, there's just no automated way to confirm it further.


## 7. Workshop creative outputs (photographic and drawn artefacts)

These are the thesis figures that are genuinely photographs or drawings rather than data charts (Figures 1, 4, 5, 6, 8). They're reproduced here as images because that's what they are; there's no raw data to regenerate them from.

In [ ]:
# NOTE: Thesis figures were originally embedded here as base64 strings.
# For GitHub-friendly file size, they have been extracted to extracted_images/thesis_figures/
# Load them back in if needed:

import os
from PIL import Image as PILImage
import matplotlib.pyplot as plt

thesis_figures_captions = {
    1: "Collective tile mosaic assembled from participants' individual creative responses.",
    4: "The written business-case artefact produced in response to the workshop's third prompt.",
    5: "The 2075 peatland collage, showing the named non-human life described above.",
    6: "'Floating gardens and foraging below', the vertically layered wet-farming imaginary described above.",
    8: "The two-register drawing, showing the farming declaration split above and below the water line.",
}

fig_dir = "extracted_images/thesis_figures"
for num, caption in thesis_figures_captions.items():
    img_path = os.path.join(fig_dir, f"figure_{num}.jpg")
    img = PILImage.open(img_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Figure {num}: {caption}", fontsize=8, wrap=True)
    plt.axis("off")
    plt.show()


## 8. Workshop photo gallery

All photographs taken during the Paludi-Imaginaries workshop, Swansea, 11 June 2026. Loaded from the `photos/` folder shipped alongside this notebook.

In [ ]:
# NOTE: Workshop photos were originally embedded here as base64 strings.
# For GitHub-friendly file size, they have been extracted to extracted_images/workshop_photos/
# Load them back in if needed:

import os
from PIL import Image as PILImage
import matplotlib.pyplot as plt

photo_dir = "extracted_images/workshop_photos"
photo_files = sorted(os.listdir(photo_dir))
n = len(photo_files)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*3.2, rows*3.2))
axes = axes.flatten()

for ax, fname in zip(axes, photo_files):
    img = PILImage.open(os.path.join(photo_dir, fname))
    ax.imshow(img)
    ax.set_title(fname, fontsize=7)
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 9. Closing discussion notes

## Paludi-Imaginaries Workshop — Closing Discussion Notes

*IUCN UK Peatland Programme Conference, Swansea, 11 June 2026 (n = 9)*

### Q1: What would it take to tell a neighbour wetland > dry land?

- Monetised value is what drives farmer decisions. Actual £ comparison needed, not abstract "wider layer" framing.
- Currently unclear wetland is worth more. Needs finance / carbon sequestration data.
- Agreement: needs to work at landscape scale, not individual farm level.
- Financial case needed if yields are equal. Resilience / climate-emission angle could help.
- Wants more open-mindedness from land management leaders. Hard to shift traditional views even with evidence.
- Preference for incremental, long-term persuasion over direct comparison ("wouldn't dive in and say this bit's better").
- Key nuance (farmer, strong point): framing shouldn't be "worth more" but "as valuable as." Comes down to the whole system: bank managers, mortgage / loan designation, supply chain, machine manufacturers, retailers. Reclassifying land as non-agricultural tanks its loan value, so a systemic economic overhaul is needed, not just a farm-level case.
- Pushback on the "worth" framing. Worth is sticky and financially loaded. Consider non-financial worth (a bog might be more fun for a kid to play in than arable land). The real question is who the neighbour is and what the actual conversation is. Also raised: we are abstracted from what land "naturally" was, everything is cultivated now, so what are we collectively calling "productive"?
- Scale point: an individual field intervention doesn't fix drainage, since the rest of the farm still needs draining. At scale (e.g. an internal drainage board) costs drop and pumping savings appear. Reverse drainage / trench funding was raised.
- Uptake example: a crop mixture project with the Gibson Institute, where a farmer WhatsApp group sharing progress photos was the key driver of uptake through peer excitement and connection.
- Facilitator reflection: farmer-forum outreach for surveys revealed organic peer discourse and a domino effect starting. Proof-of-concept from a peer-to-peer example ("my mate's doing it down the road") was seen as key to paludiculture adoption.

### Q2: Did your thinking or feeling on peatlands shift during the session?

- Yes: shifted from purely quantifying / numbers-based thinking to a more artistic, felt understanding.
- The group jokingly credited Felix for this shift.
- Became more curious about paludiculture. Understanding "farm on wet soil ≠ flood" was the key unlock, enabling concrete imagining (e.g. distillery table example).
- Water table management raised as the central challenge. It isn't agronomy or will that's missing, it's engineering scalability and commercial viability with consistency.
- A few "maybe it changed" responses.
- One noted a mindset shift regarding grazing / land use ease.
- One said no fundamental shift: the sites already felt too extracted, and change was seen as improvement rather than loss.
- Generational angle raised: need to think about engaging the next generation of land managers. Tension between large connected landscape-scale restoration and the small local pockets people actually engage with day to day. Multiple scales matter.


### Q3: What surprised you in the creative process?

- Realisation: need to do more art (facilitator).
- Surprised how much participants struggled to love agricultural landscapes.
- Simply nice and restful to draw, described as "a respite."
- One participant: the process became about combining "what we have now" with "where we want to be," drawing aimed at connecting and integrating farming with nature rather than separating them, with nature as the overarching goal.
- Facilitator: the creative / speculative lens forces conceptualising things that don't yet exist. "Future happens because we decide it." The paludiculture project is about imagining wet-farming futures.
- One participant described a long personal and professional history with land since 2011. The creative reflection prompted deeper personal reflection.
- Surprised by the feelings and memories surfaced by the drawing exercise.
- One participant was surprised just by watching others' creative process.
- One participant was surprised they were even able to draw at all.

### Closing notes (facilitator's own reflections)

- Valued the networking and relationship-building that came from the session.
- This was the first workshop facilitated, and there was some nervousness. The classroom-style room setup was reconfigured on the morning of the session to avoid a rigid, separated table layout.
- Thanked participants for their consent forms and engagement.


## 10. Notes on data provenance, item coding, and correction log

- **Survey data**: `Paludiculture_Stakeholder_Survey__Landscape__Language_and_the_Barriers_to_Wet_Farming__1-52.xlsx`, n = 52 respondents, embedded in Section 1 above in full.
- **Workshop photographs**: 33 images from the Paludi-Imaginaries workshop, Swansea, 11 June 2026, converted from HEIC to JPEG and embedded in Section 8 above.
- **Closing discussion notes**: facilitator notes from the workshop's closing discussion (Q1–Q3 plus facilitator reflections), reproduced in full in Section 9.
- **Item coding**: the imagination gap / trust gap matrix in Section 3, and every statistical test in Section 6, score imaginative clarity from Q12 ("I can picture what a thriving paludiculture landscape would look like," a 5-point Likert item) and economic trust from Q11 (a 4-point single-select item), exactly as specified in the thesis's Methodology §3.2. Clarity is coded "clear" for Agree/Strongly agree and not-clear otherwise, including "Neither." Trust is coded "trusting" for Moderately/Trust it fully. Q10, a related but coarser single-select item ("how clear is that picture?"), is retained in the data (`clarity_score`) but not used for any thesis-matching statistic.
- **Barrier coding**: "Major barrier" is the sole positive category throughout Section 6. Non-respondents to a given barrier item are excluded listwise from that item's calculations, consistent with how the barrier percentages themselves are reported.

— see Section 6 above for the full verification.